### Basic Thread Creation

In [11]:
import threading
import time

def print_numbers(start, end, thread_id):
    for i in range(start, end + 1):
        print(f"Thread{thread_id} : {i}")
        time.sleep(1)

thread1 = threading.Thread(target=print_numbers, args=(1, 5, 1))
thread2 = threading.Thread(target=print_numbers, args=(6, 10, 2))
threads = [thread1, thread2]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()

Thread1 : 1
Thread2 : 6
Thread1 : 2Thread2 : 7

Thread2 : 8Thread1 : 3

Thread1 : 4Thread2 : 9

Thread2 : 10Thread1 : 5



### Race Conditions and Shared Data

In [42]:
import threading
import time

def increment_and_print_counter():
    global counter
    for _ in range(10000):
        temp = counter
        time.sleep(0.0001)
        counter = temp + 1

counter = 0

thread1 = threading.Thread(target=increment_and_print_counter)
thread2 = threading.Thread(target=increment_and_print_counter)
threads = [thread1, thread2]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(counter)

10000


### Fix the race condition using Locks

In [44]:
import threading
import time

lock = threading.Lock()

def increase_counter():
    global counter
    for _ in range(10000):
        with lock:
            # print(f"Thread ID: {threading.get_ident()}")
            temp = counter
            time.sleep(0.0001)
            counter = temp + 1

counter = 0
threads = [threading.Thread(target=increase_counter) for _ in range(5)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(counter)

50000


### Using thread.Event to Signal Between Threads

In [53]:
import threading
import time

event = threading.Event()

def thread_func(thread_id):
    global event
    print(f"Thread {thread_id} started.")
    time.sleep(5)
    print(f"Thread {thread_id} completed.")
    if thread_id == 0:
        event.set()

threads = [threading.Thread(target=thread_func, args=(i,)) for i in range(2)]
threads[0].start()
event.wait()
threads[1].start()
for thread in threads:
    thread.join()

Thread 0 started.
Thread 0 completed.
Thread 1 started.
Thread 1 completed.


### Challenge 5: Producer-Consumer Using queue.Queue

In [78]:
import threading
import queue
import time
q = queue.Queue()

def produce_message(num):
    global q
    q.put(num)
    print(f"Produced: {num}")
    time.sleep(0.001)

def consume_message():
    global q
    while True:
        num = q.get()
        if num is None:
            q.task_done()
            break
        print(f"Consumed: {num}")
        q.task_done()
            
producer_threads = [threading.Thread(target=produce_message, args=(i,)) for i in range(5)]
consumer_threads = [threading.Thread(target=consume_message) for _ in range(5)]
for thread in producer_threads:
    thread.start()
for _ in range(len(consumer_threads)):
    q.put(None)
for thread in producer_threads:
    thread.join()
for thread in consumer_threads:
    thread.start()
for thread in consumer_threads:
    thread.join()

Produced: 0
Produced: 1
Produced: 2
Produced: 3
Produced: 4
Consumed: 0
Consumed: 1
Consumed: 2
Consumed: 3
Consumed: 4


### Basic Thread Pool executor

In [82]:
from concurrent.futures import ThreadPoolExecutor
import time

def worker(n):
    print(f"Processing: {n}")
    time.sleep(1)
    return f"Done {n}"

# Cretes a pool with three workers
# Uses context manager for clean up
with ThreadPoolExecutor(max_workers=3) as executor:
    # Runs worker(n) on 5 inputs, distributing them across the pool
    results = executor.map(worker, range(5))

for result in results:
    print(result)

Processing: 0
Processing: 1
Processing: 2
Processing: 3Processing: 4

Done 0
Done 1
Done 2
Done 3
Done 4


### Futures

In [84]:
from concurrent.futures import ThreadPoolExecutor
import time

def worker(n):
    time.sleep(2)
    return f"Task {n} done"

with ThreadPoolExecutor() as executor:
    future = executor.submit(worker, 1) # Returns a future object

    print(future.done()) # False (Task is still running)
    print(future.result()) # Blocks until the result is available
    print(future.done()) # True (Task is completed)

False
Task 1 done
True


### Handling Exceptions in Future

In [86]:
from concurrent.futures import ThreadPoolExecutor
import time

def faulty_task():
    time.sleep(2)
    raise ValueError(f"Something went wrong")

with ThreadPoolExecutor() as executor:
    future = executor.submit(faulty_task)

    try:
        future.result() # Raises ValueError
    except ValueError as e:
        print(f"{e}")

Something went wrong


### Using as_completed() with Futures

In [93]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

def worker(n):
    time.sleep(1 / n)
    return f"Task {n} done"

with ThreadPoolExecutor() as executor:
    futures = [executor.submit(worker, i) for i in range(1, 4)]

    for future in as_completed(futures):
        print(future.result())

Task 3 done
Task 2 done
Task 1 done


### Single-Threaded vs. Multi-Threaded Performance

In [96]:
import threading
import time

def worker(n):
    time.sleep(n)
    print(f"Task {n} done")

start_time = time.time()
for i in range(5):
    worker(i)
print(f"Sequential processing took: {time.time() - start_time} seconds.")

start_time = time.time()
threads = [threading.Thread(target=worker, args=(i,)) for i in range(5)]
for thread in threads:
    thread.start()
for thread in threads:
    thread.join()
print(f"Thread processing took: {time.time() - start_time} seconds.")

Task 0 done
Task 1 done
Task 2 done
Task 3 done
Task 4 done
Sequential processing took: 10.017055034637451 seconds.
Task 0 done
Task 1 done
Task 2 done
Task 3 done
Task 4 done
Thread processing took: 4.00754189491272 seconds.
